# Exploratory Data Analysis

**Dataset**: [TODO: Dataset name]
**Author**: [TODO: Your name]
**Date**: [TODO: Date]

## Objective
[TODO: Describe the analysis objective]

## 1. Setup

In [ ]:
# Standard library
import os
import sys
from pathlib import Path

# Data handling
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration
SEED = 42
np.random.seed(SEED)

# Display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

# Print versions
print(f"Python: {sys.version}")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")

## 2. Data Loading

In [ ]:
# TODO: Update path to your data
DATA_PATH = Path('data/raw')
df = pd.read_csv(DATA_PATH / 'dataset.csv')

print(f"Loaded {len(df):,} rows, {len(df.columns)} columns")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1e6:.2f} MB")

## 3. Basic Information

In [ ]:
# Data types and non-null counts
print(df.info())

In [ ]:
# First few rows
df.head()

In [ ]:
# Last few rows
df.tail()

In [ ]:
# Random sample
df.sample(5, random_state=SEED)

In [ ]:
# Descriptive statistics for numeric columns
df.describe()

In [ ]:
# Descriptive statistics for categorical columns
df.describe(include=['object', 'category'])

## 4. Missing Values Analysis

In [ ]:
# Missing value summary
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({
    'missing_count': missing,
    'missing_pct': missing_pct
}).query('missing_count > 0').sort_values('missing_pct', ascending=False)

if len(missing_df) > 0:
    print(f"Columns with missing values: {len(missing_df)}")
    display(missing_df)
else:
    print("No missing values found!")

In [ ]:
# Visualize missing values
if len(missing_df) > 0:
    plt.figure(figsize=(10, 6))
    sns.barplot(data=missing_df.reset_index(), x='index', y='missing_pct')
    plt.xticks(rotation=45, ha='right')
    plt.xlabel('Column')
    plt.ylabel('Missing %')
    plt.title('Missing Values by Column')
    plt.tight_layout()
    plt.show()

## 5. Univariate Analysis

### 5.1 Numeric Features

In [ ]:
# Identify numeric columns
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"Numeric columns ({len(numeric_cols)}): {numeric_cols}")

In [ ]:
# Histograms for numeric columns
n_cols = min(len(numeric_cols), 12)  # Limit to 12 columns
if n_cols > 0:
    fig, axes = plt.subplots(
        nrows=(n_cols + 2) // 3, 
        ncols=3, 
        figsize=(15, 4 * ((n_cols + 2) // 3))
    )
    axes = axes.flatten()
    
    for idx, col in enumerate(numeric_cols[:n_cols]):
        df[col].hist(ax=axes[idx], bins=30, edgecolor='black', alpha=0.7)
        axes[idx].set_title(col)
        axes[idx].set_xlabel('')
    
    # Hide unused subplots
    for idx in range(n_cols, len(axes)):
        axes[idx].set_visible(False)
    
    plt.suptitle('Numeric Feature Distributions', y=1.02, fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
# Box plots for outlier detection
if len(numeric_cols) > 0:
    n_cols = min(len(numeric_cols), 12)
    fig, axes = plt.subplots(
        nrows=(n_cols + 2) // 3, 
        ncols=3, 
        figsize=(15, 4 * ((n_cols + 2) // 3))
    )
    axes = axes.flatten()
    
    for idx, col in enumerate(numeric_cols[:n_cols]):
        sns.boxplot(data=df, y=col, ax=axes[idx])
        axes[idx].set_title(col)
    
    for idx in range(n_cols, len(axes)):
        axes[idx].set_visible(False)
    
    plt.suptitle('Numeric Feature Box Plots (Outlier Detection)', y=1.02, fontsize=14)
    plt.tight_layout()
    plt.show()

### 5.2 Categorical Features

In [ ]:
# Identify categorical columns
categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
print(f"Categorical columns ({len(categorical_cols)}): {categorical_cols}")

In [ ]:
# Value counts for categorical columns
for col in categorical_cols:
    print(f"\n{col} ({df[col].nunique()} unique values):")
    print(df[col].value_counts().head(10))

In [ ]:
# Bar plots for categorical columns (low cardinality only)
low_cardinality = [col for col in categorical_cols if df[col].nunique() <= 10]

if len(low_cardinality) > 0:
    n_cols = min(len(low_cardinality), 6)
    fig, axes = plt.subplots(
        nrows=(n_cols + 1) // 2, 
        ncols=2, 
        figsize=(14, 4 * ((n_cols + 1) // 2))
    )
    axes = axes.flatten()
    
    for idx, col in enumerate(low_cardinality[:n_cols]):
        df[col].value_counts().plot(kind='bar', ax=axes[idx], edgecolor='black')
        axes[idx].set_title(col)
        axes[idx].set_xlabel('')
        plt.setp(axes[idx].xaxis.get_majorticklabels(), rotation=45, ha='right')
    
    for idx in range(n_cols, len(axes)):
        axes[idx].set_visible(False)
    
    plt.suptitle('Categorical Feature Distributions', y=1.02, fontsize=14)
    plt.tight_layout()
    plt.show()

## 6. Bivariate Analysis

### 6.1 Target Variable Analysis

[TODO: Update TARGET_COL to your target variable]

In [ ]:
# TODO: Set your target variable
TARGET_COL = 'target'  # Change this

if TARGET_COL in df.columns:
    print(f"Target variable: {TARGET_COL}")
    print(f"Type: {df[TARGET_COL].dtype}")
    print(f"\nValue counts:")
    print(df[TARGET_COL].value_counts())
    
    # Plot target distribution
    plt.figure(figsize=(8, 5))
    if df[TARGET_COL].dtype in ['object', 'category'] or df[TARGET_COL].nunique() <= 10:
        df[TARGET_COL].value_counts().plot(kind='bar', edgecolor='black')
        plt.ylabel('Count')
    else:
        df[TARGET_COL].hist(bins=30, edgecolor='black')
        plt.ylabel('Frequency')
    plt.title(f'Target Distribution: {TARGET_COL}')
    plt.xlabel(TARGET_COL)
    plt.tight_layout()
    plt.show()
else:
    print(f"Target column '{TARGET_COL}' not found in dataset")

### 6.2 Feature vs Target

In [ ]:
# Numeric features vs target (if target is categorical)
if TARGET_COL in df.columns and df[TARGET_COL].nunique() <= 10:
    for col in numeric_cols[:6]:  # Limit to first 6
        if col != TARGET_COL:
            plt.figure(figsize=(10, 5))
            sns.boxplot(data=df, x=TARGET_COL, y=col)
            plt.title(f'{col} by {TARGET_COL}')
            plt.tight_layout()
            plt.show()

## 7. Correlation Analysis

In [ ]:
# Correlation matrix for numeric columns
if len(numeric_cols) > 1:
    corr_matrix = df[numeric_cols].corr()
    
    plt.figure(figsize=(12, 10))
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    sns.heatmap(
        corr_matrix, 
        mask=mask,
        annot=True if len(numeric_cols) <= 15 else False,
        fmt='.2f',
        cmap='coolwarm',
        center=0,
        square=True,
        linewidths=0.5
    )
    plt.title('Feature Correlation Matrix')
    plt.tight_layout()
    plt.show()

In [ ]:
# Highly correlated features
if len(numeric_cols) > 1:
    high_corr = []
    for i in range(len(corr_matrix.columns)):
        for j in range(i + 1, len(corr_matrix.columns)):
            if abs(corr_matrix.iloc[i, j]) > 0.7:
                high_corr.append({
                    'feature_1': corr_matrix.columns[i],
                    'feature_2': corr_matrix.columns[j],
                    'correlation': corr_matrix.iloc[i, j]
                })
    
    if high_corr:
        print("Highly correlated feature pairs (|r| > 0.7):")
        display(pd.DataFrame(high_corr).sort_values('correlation', key=abs, ascending=False))
    else:
        print("No highly correlated feature pairs found.")

In [ ]:
# Correlation with target
if TARGET_COL in df.columns and TARGET_COL in numeric_cols:
    target_corr = corr_matrix[TARGET_COL].drop(TARGET_COL).sort_values(key=abs, ascending=False)
    
    plt.figure(figsize=(10, 6))
    target_corr.plot(kind='barh')
    plt.xlabel('Correlation with Target')
    plt.title(f'Feature Correlations with {TARGET_COL}')
    plt.tight_layout()
    plt.show()

## 8. Key Findings Summary

### Data Overview
- **Rows**: [TODO]
- **Columns**: [TODO]
- **Numeric features**: [TODO]
- **Categorical features**: [TODO]

### Missing Values
[TODO: Summarize missing value findings]

### Target Variable
[TODO: Describe target distribution, class imbalance if applicable]

### Key Correlations
[TODO: List important correlations]

### Potential Issues
- [TODO: List any data quality issues]
- [TODO: List outliers]
- [TODO: List class imbalance]

### Recommendations for Modeling
1. [TODO: Preprocessing suggestions]
2. [TODO: Feature engineering ideas]
3. [TODO: Modeling approach suggestions]

In [ ]:
# Save processed data (optional)
# df.to_csv(DATA_PATH / 'processed' / 'dataset_cleaned.csv', index=False)